---
## 5. Cliente Python

Ejecuta estas celdas desde **otro kernel o máquina** mientras el servidor está corriendo.

Si estás en LightningAI, reemplaza `SERVER_URL` por la URL pública del puerto.

In [ ]:
import requests

SERVER_URL = f"http://0.0.0.0:8000"   # o https://<id>-{PORT}.cloudspaces.litng.ai

# Health check
resp = requests.get(f"{SERVER_URL}/health")
print("Health:", resp.json())

# Info del modelo
resp = requests.get(f"{SERVER_URL}/model")
print("Model: ", resp.json())

Health: {'status': 'ok', 'model_loaded': True, 'device': 'cuda'}
Model:  {'hub_repo': 'jezapataf/tweeteval-emotion-distilbert', 'labels': ['anger', 'joy', 'optimism', 'sadness'], 'parameters': '67.0M', 'device': 'cuda'}


In [ ]:
# Predicción de un solo tweet
tweet = "I can't believe this is happening, I'm devastated 😢"

resp = requests.post(
    f"{SERVER_URL}/predict",
    json={"text": tweet},
)
resp.raise_for_status()

result = resp.json()
print(f"Tweet: {result['text']}\n")
for p in result["predictions"]:
    bar = "█" * int(p["score"] * 40)
    print(f"  {p['label']:<12} {p['score']:.2%}  {bar}")

Tweet: I can't believe this is happening, I'm devastated 😢

  sadness      92.64%  █████████████████████████████████████
  anger        4.51%  █
  joy          1.66%  
  optimism     1.19%  


In [ ]:
# Predicción en batch
tweets = [
    "I'm so happy today!!! 😊",
    "this is absolutely disgusting #angry #wtf",
    "@user can't believe what just happened... 😢",
    "The future looks bright #hope #goals",
]

resp = requests.post(
    f"{SERVER_URL}/predict/batch",
    json={"texts": tweets},
)
resp.raise_for_status()

print(f"{'Tweet':<50}  {'Emoción':<12}  {'Score':>6}")
print("-" * 74)
for item in resp.json()["results"]:
    top     = item["predictions"][0]
    preview = (item["text"][:47] + "...") if len(item["text"]) > 50 else item["text"].ljust(50)
    print(f"{preview}  {top['label']:<12}  {top['score']:.4f}")

Tweet                                               Emoción        Score
--------------------------------------------------------------------------
I'm so happy today!!! 😊                             joy           0.9464
this is absolutely disgusting #angry #wtf           anger         0.9789
@user can't believe what just happened... 😢         sadness       0.8002
The future looks bright #hope #goals                optimism      0.8056


## 6. Script cliente standalone

Este script puede correrse desde cualquier máquina que tenga acceso al servidor.

In [4]:
client_script = '''
#!/usr/bin/env python
"""Cliente para el servidor de clasificación de emociones.

Uso:
    python client.py "I'm so happy today!"
    python client.py "I'm so happy today!" --url https://<id>-8081.cloudspaces.litng.ai
    python client.py --batch tweets.txt
"""
import argparse
import requests


def predict_one(url: str, text: str) -> None:
    resp = requests.post(f"{url}/predict", json={"text": text})
    resp.raise_for_status()
    result = resp.json()
    print(f"\nTweet: {result['text']}\n")
    for p in result["predictions"]:
        bar = "█" * int(p["score"] * 40)
        print(f"  {p['label']:<12} {p['score']:.2%}  {bar}")


def predict_batch(url: str, filepath: str) -> None:
    with open(filepath) as f:
        texts = [line.strip() for line in f if line.strip()]

    resp = requests.post(f"{url}/predict/batch", json={"texts": texts})
    resp.raise_for_status()

    print(f"{\"Tweet\":<50}  {\"Emoción\":<12}  {\"Score\":>6}")
    print("-" * 74)
    for item in resp.json()["results"]:
        top     = item["predictions"][0]
        preview = (item["text"][:47] + "...") if len(item["text"]) > 50 else item["text"].ljust(50)
        print(f"{preview}  {top[\'label\']:<12}  {top[\'score\']:.4f}")


def main() -> None:
    parser = argparse.ArgumentParser(description="Emotion classifier client")
    parser.add_argument("text",    nargs="?", help="Texto a clasificar")
    parser.add_argument("--batch", metavar="FILE", help="Archivo con un tweet por línea")
    parser.add_argument("--url",   default="http://localhost:8081", help="URL del servidor")
    args = parser.parse_args()

    if args.batch:
        predict_batch(args.url, args.batch)
    elif args.text:
        predict_one(args.url, args.text)
    else:
        parser.print_help()


if __name__ == "__main__":
    main()
'''

with open("client.py", "w") as f:
    f.write(client_script.strip())

print("client.py guardado.")
print("\nUso:")
print('  python client.py "I am so happy today!"')
print('  python client.py "I am devastated 😢" --url https://<id>-8081.cloudspaces.litng.ai')
print('  python client.py --batch tweets.txt')

client.py guardado.

Uso:
  python client.py "I am so happy today!"
  python client.py "I am devastated 😢" --url https://<id>-8081.cloudspaces.litng.ai
  python client.py --batch tweets.txt
